<a href="https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbinaqeel-analyst/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [7]:
con.sql(f"""
SELECT COUNT(*) AS rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌─────────┐
│  rows   │
│  int64  │
├─────────┤
│ 9841378 │
└─────────┘

## 1. Rule

My baseline ranks pages for refresh review using two observed signals available during the decision period: search visibility and click-through performance adjusted for search position.

First, I check whether pages with more impressions provide a meaningful opportunity for review. A page with very low visibility may have too little evidence for a reliable action decision, while a visible page with weak click capture can potentially be worth reviewing.

Second, I compare CTR within position tiers rather than comparing all pages directly. Pages ranking in different positions naturally receive different CTR, so a low CTR is more meaningful when compared with pages that have similar search positions.

My rule gives higher priority to pages with sufficient impressions, relatively strong visibility, and CTR below the typical level for their position tier. The output is a decision-support queue, not a prediction that refreshing a page will improve performance.

Primary reason code: HIGH_VISIBILITY_LOW_CTR

Action label: REVIEW_FOR_REFRESH

The rule uses only observed March 2026 signals available at the decision moment. It does not use future performance, product flags, or a label derived from a later outcome.

Signal check 1 — Visibility / impressions: MIXED

Across the five impression buckets (n = 176,738 pages), higher-visibility pages generally had higher median CTR and better median positions. For example, the highest-impression bucket had median impressions of 3,932, median CTR of 0.0020, and median position of 6.47, compared with median impressions of 3, median CTR of 0.0000, and median position of 7.00 in the lowest bucket.

This does not show that high impressions alone mean a page should be refreshed. However, it confirms that higher-visibility pages represent a larger potential opportunity and provide more evidence than very low-volume pages. I will therefore use impressions as an eligibility and priority signal, not as proof that a refresh is needed.

Signal check 2 — CTR versus position: CONFIRMED

Across 176,738 pages, mean CTR decreased consistently as average position became worse: 0.0124 for positions 1–3, 0.0049 for positions 4–10, 0.0032 for positions 11–20, 0.0023 for positions 21–50, and 0.0009 for positions above 50.

This confirms that CTR should be interpreted relative to position rather than compared equally across all pages. This is the flag-linked signal behind my baseline rule.

In [8]:
import pandas as pd
import numpy as np

# Read only the March 2026 decision-time slice
df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

# Aggregate daily rows so the baseline eventually ranks pages,
# not individual page-day rows.
page_df = (
    df.groupby(["client_hash_id", "content_hash_id"], as_index=False)
      .agg(
          impressions=("gsc_impressions", "sum"),
          clicks=("gsc_clicks", "sum"),
          avg_position=("gsc_avg_position", "mean"),
          days_observed=("report_date", "nunique")
      )
)

# Calculate March CTR
page_df["ctr"] = np.where(
    page_df["impressions"] > 0,
    page_df["clicks"] / page_df["impressions"],
    np.nan
)

# Keep pages with meaningful visibility for the checks
visible = page_df[page_df["impressions"] > 0].copy()

print("SIGNAL CHECK 1 — VISIBILITY / IMPRESSIONS")
print("n =", len(visible))

visible["impression_bucket"] = pd.qcut(
    visible["impressions"],
    q=5,
    duplicates="drop"
)

impression_check = (
    visible.groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
)

print(impression_check.round(4))


# Position tiers for fair CTR comparison
def position_tier(pos):
    if pos <= 3:
        return "1-3"
    elif pos <= 10:
        return "4-10"
    elif pos <= 20:
        return "11-20"
    elif pos <= 50:
        return "21-50"
    return "51+"

visible["position_tier"] = visible["avg_position"].apply(position_tier)

print("\nSIGNAL CHECK 2 — CTR BY POSITION TIER")
print("n =", len(visible))

ctr_check = (
    visible.groupby("position_tier", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_position=("avg_position", "median"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        median_impressions=("impressions", "median")
    )
)

# Put tiers in search-order
tier_order = ["1-3", "4-10", "11-20", "21-50", "51+"]
ctr_check = ctr_check.reindex(
    [x for x in tier_order if x in ctr_check.index]
)

print(ctr_check.round(4))

# Keep the prepared page-level dataframe for the next section
page_df = visible.copy()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL CHECK 1 — VISIBILITY / IMPRESSIONS
n = 176738
                        n  median_impressions  median_ctr  median_position
impression_bucket                                                         
(0.999, 11.0]       36167                 3.0      0.0000           7.0000
(11.0, 81.0]        34732                34.0      0.0000          11.2606
(81.0, 351.0]       35208               174.0      0.0000          12.8919
(351.0, 1538.0]     35292               716.0      0.0014           8.9008
(1538.0, 617124.0]  35339              3932.0      0.0020           6.4726

SIGNAL CHECK 2 — CTR BY POSITION TIER
n = 176738
                   n  median_position  mean_ctr  median_ctr  \
position_tier                                                 
1-3            17578           2.1019    0.0124         0.0   
4-10           81988           6.0225    0.0049         0.0   
11-20          32203          13.9399    0.0032         0.0   
21-50          33288          29.6389    0.0023         0

## 2. Build the ranked queue (writes the CSV)

My baseline first limits the queue to pages with meaningful March visibility. I use 351 impressions as the minimum because this is the start of the upper two impression buckets in my signal check.

For each eligible page, I compare its CTR with the mean CTR of pages in the same position tier. Pages below their tier's observed mean CTR receive a positive CTR opportunity gap.

The baseline score combines visibility with this CTR gap:

baseline score = log(1 + impressions) × max(expected CTR for position tier − actual CTR, 0)

Higher scores represent pages with both substantial visibility and a larger observed CTR gap relative to pages at similar positions.

Every selected page receives the reason code HIGH_VISIBILITY_LOW_CTR and the action label REVIEW_FOR_REFRESH. This is a prioritization rule for review, not a prediction that refreshing the page will improve performance.

In [9]:
import os
import numpy as np
import pandas as pd

# Work from the page-level March dataframe created in Section 1
queue = page_df.copy()

# Recalculate position tier explicitly
def position_tier(pos):
    if pos <= 3:
        return "1-3"
    elif pos <= 10:
        return "4-10"
    elif pos <= 20:
        return "11-20"
    elif pos <= 50:
        return "21-50"
    return "51+"

queue["position_tier"] = queue["avg_position"].apply(position_tier)

# Observed expected CTR within each comparable position tier
tier_expected_ctr = (
    queue.groupby("position_tier")["ctr"]
    .mean()
    .to_dict()
)

queue["expected_ctr"] = queue["position_tier"].map(tier_expected_ctr)

# Only pages with meaningful visibility are eligible for review
queue["eligible"] = queue["impressions"] >= 351

# Positive gap means the page captures fewer clicks than the
# observed mean for its position tier
queue["ctr_gap"] = (
    queue["expected_ctr"] - queue["ctr"]
).clip(lower=0)

# Transparent baseline score:
# larger visibility × larger underperformance relative to position peers
queue["baseline_score"] = np.where(
    queue["eligible"],
    np.log1p(queue["impressions"]) * queue["ctr_gap"],
    0
)

# One reason code and one action label
queue["reason_code"] = np.where(
    (queue["eligible"]) & (queue["ctr_gap"] > 0),
    "HIGH_VISIBILITY_LOW_CTR",
    "NOT_PRIORITIZED"
)

queue["action_label"] = np.where(
    (queue["eligible"]) & (queue["ctr_gap"] > 0),
    "REVIEW_FOR_REFRESH",
    "MONITOR"
)

# Rank the review candidates
ranked_queue = (
    queue[queue["reason_code"] == "HIGH_VISIBILITY_LOW_CTR"]
    .sort_values(
        ["baseline_score", "impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = ranked_queue.index + 1

# Keep useful review columns
output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_tier",
    "expected_ctr",
    "ctr_gap",
    "baseline_score",
    "reason_code",
    "action_label"
]

ranked_queue = ranked_queue[output_cols]

# Write the required CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_path, index=False)

print(f"Total March pages: {len(queue):,}")
print(f"Eligible pages (>= 351 impressions): {queue['eligible'].sum():,}")
print(f"Review candidates: {len(ranked_queue):,}")
print(f"\nSaved to: {output_path}")

ranked_queue.head(20)

Total March pages: 176,738
Eligible pages (>= 351 impressions): 70,715
Review candidates: 57,089

Saved to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_tier,expected_ctr,ctr_gap,baseline_score,reason_code,action_label
0,1,client_e547b89c05043229,content_306bc78dff1eb683,80821,35,0.000433,1.488604,1-3,0.012399,0.011966,0.135220,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
1,2,client_e547b89c05043229,content_8d7d99f109e19aa2,203497,289,0.001420,2.563756,1-3,0.012399,0.010979,0.134204,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
2,3,client_62f4a7e64f5e0096,content_fc67675904376267,60172,18,0.000299,2.261303,1-3,0.012399,0.012100,0.133163,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
3,4,client_e547b89c05043229,content_c46df0fa61530d86,70398,42,0.000597,1.556258,1-3,0.012399,0.011803,0.131742,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
4,5,client_1a730cb2640a1abf,content_d61fc394d10cba41,38000,1,0.000026,2.740744,1-3,0.012399,0.012373,0.130479,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
5,6,client_e547b89c05043229,content_9ef3d7516483e665,89229,92,0.001031,2.481596,1-3,0.012399,0.011368,0.129588,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
6,7,client_e547b89c05043229,content_b2b85c287474668d,65304,61,0.000934,1.541702,1-3,0.012399,0.011465,0.127114,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
7,8,client_62f4a7e64f5e0096,content_7f52754cb72a5991,43135,28,0.000649,2.502348,1-3,0.012399,0.011750,0.125401,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
8,9,client_73cda7b4e4f265ea,content_252aa5480bb1f8d7,66698,75,0.001124,2.394123,1-3,0.012399,0.011275,0.125242,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH
9,10,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,25941,3,0.000116,2.418270,1-3,0.012399,0.012284,0.124848,HIGH_VISIBILITY_LOW_CTR,REVIEW_FOR_REFRESH


## 3. Top-20 review

3. Top-20 review

All top 20 pages were ranked as REVIEW_FOR_REFRESH with the reason code HIGH_VISIBILITY_LOW_CTR. They are highly visible pages in the 1–3 position tier whose observed CTR is substantially below the average CTR of that tier. This makes them candidates for review, but not automatic proof that refreshing the content will improve performance.

Rank	Action	Why it is here	Confidence	What would make it wrong?
1	Review for refresh	Very high impressions and CTR far below the 1–3 tier average	High	The low CTR may be caused by SERP features or query intent rather than page content

2	Review for refresh	Highest impression volume among the top few pages with a large CTR gap	High	The page may already match user intent, while the SERP itself reduces clicks

3	Review for refresh	High visibility with an extremely large gap between expected and observed CTR	High	Position or CTR may vary substantially by query, which the aggregate score hides

4	Review for refresh	Large impression volume and very low CTR despite a strong average position	High	A title or meta change may not address the real cause of low clicks

5	Review for refresh	Strong visibility and almost no clicks, producing one of the largest CTR gaps	High	The impressions may come from queries where users do not need to click a result

6	Review for refresh	High impressions with CTR well below the expected value for positions 1–3	High	The low CTR could be temporary or driven by unusual query mix

7	Review for refresh	High visibility and a large CTR gap at a very strong average position	High	Average position may hide different performance across individual queries

8	Review for refresh	Large impression volume with CTR far below its position-tier benchmark	High	The benchmark may not fully account for client or content-type differences

9	Review for refresh	High impressions and a substantial opportunity gap between expected and observed CTR	High	The page may be competing with rich results or other SERP elements

10	Review for refresh	Strong position and visibility but almost no clicks	High	Low CTR may reflect informational queries answered directly in search

11	Review for refresh	Large CTR gap with enough impressions to make the signal worth reviewing	High	The pattern may not persist in another time window

12	Review for refresh	Strong position and very low CTR relative to the tier average	High	A refresh could waste effort if the problem is outside the page itself

13	Review for refresh	High visibility with a large difference between expected and actual CTR	High	The expected CTR average may not be appropriate for this page's query mix

14	Review for refresh	Good impression volume and almost zero CTR despite position near the top	High	Searchers may prefer another result because of brand or SERP context

15	Review for refresh	High impressions and a large CTR opportunity gap	High	The observed CTR could be affected by short-term fluctuations

16	Review for refresh	Strong visibility and CTR substantially below the position-tier expectation	High	The page may not actually need content changes after manual review

17	Review for refresh	Enough impressions and a very large CTR gap to justify investigation	Medium-High	The aggregate position may conceal poor positions for the highest-volume queries

18	Review for refresh	Strong ranking position but CTR remains far below the expected tier level	Medium-High	Query intent or SERP layout could explain the gap better than page quality

19	Review for refresh	Large CTR gap with substantial impression volume	Medium-High	The benchmark does not control for all factors affecting click behavior

20	Review for refresh	High visibility and CTR well below the expected level for its position tier	Medium-High	A manual review could show that no useful refresh action is available

Overall review: The strongest picks are the pages with both very high impression volume and a large CTR gap because a possible improvement would affect more search exposures. However, this baseline is a decision-support queue, not proof that every page needs a refresh or that changing the page will increase clicks. The top-ranked pages should be manually reviewed for query intent, SERP features, title/snippet quality, and content relevance before taking action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

A weakness of this baseline is that all top-ranked pages received the same reason code, HIGH_VISIBILITY_LOW_CTR. The score is mainly driven by impression volume and the gap between observed CTR and the average CTR for the page's position tier. This can produce weak picks when low CTR is caused by factors outside the page itself, such as query intent, SERP features, or differences hidden by an average position.

I did not use product flags or any future-window outcomes in the baseline score. The features used for ranking were calculated from the March 2026 slice only: impressions, clicks, CTR, and average position. The expected CTR was calculated from the same March 2026 position tiers. Therefore, the baseline does not use a later outcome or label-derived feature to rank pages.

This is still a decision-support baseline, not evidence that refreshing a page will cause CTR to improve.

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
print("DATA WINDOW CHECK")
print("Minimum date:", df["report_date"].min())
print("Maximum date:", df["report_date"].max())

print("\nFEATURES USED IN BASELINE")
print([
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_tier",
    "expected_ctr",
    "ctr_gap"
])

print("\nLEAKAGE CHECK")
print("No future-window columns were used.")
print("No product flags were used.")
print("The baseline was built only from the March 2026 data slice.")

DATA WINDOW CHECK
Minimum date: 2026-03-01 00:00:00
Maximum date: 2026-03-31 00:00:00

FEATURES USED IN BASELINE
['impressions', 'clicks', 'ctr', 'avg_position', 'position_tier', 'expected_ctr', 'ctr_gap']

LEAKAGE CHECK
No future-window columns were used.
No product flags were used.
The baseline was built only from the March 2026 data slice.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.